# Ex4 — Polars: mesma lógica, nova filosofia

**Objetivo:** Reproduzir as mesmas transformações do Ex3 (Pandas) usando a API do Polars.  
O foco é perceber as diferenças de design entre as duas bibliotecas.

Conceitos avaliados:
- `with_columns()` — equivalente ao `.assign()`
- `filter()` com `pl.col()` — equivalente ao `.query()`
- `group_by().agg()` — sintaxe Polars
- Expressões de window com `over()` — equivalente ao `.transform()`
- `sort()` + `group_by()` para substituir `idxmax()`

In [5]:
import polars as pl

vendas = pl.DataFrame([
    {"pedido": 1,  "cliente": "João",    "vendedor": "Bruno",  "produto": "Notebook",   "categoria": "Eletrônicos", "qtd": 1, "preco": 4500.0, "status": "entregue"},
    {"pedido": 2,  "cliente": "Maria",   "vendedor": "Carla",  "produto": "Mouse",      "categoria": "Periféricos", "qtd": 3, "preco":   89.9, "status": "entregue"},
    {"pedido": 3,  "cliente": "João",    "vendedor": "Bruno",  "produto": "SSD 1TB",   "categoria": "Componentes", "qtd": 2, "preco":  380.0, "status": "entregue"},
    {"pedido": 4,  "cliente": "Ana",     "vendedor": "Diego",  "produto": "Headset",   "categoria": "Periféricos", "qtd": 1, "preco":  450.0, "status": "cancelado"},
    {"pedido": 5,  "cliente": "Pedro",   "vendedor": "Bruno",  "produto": "Monitor 4K","categoria": "Eletrônicos", "qtd": 1, "preco": 2800.0, "status": "entregue"},
    {"pedido": 6,  "cliente": "Maria",   "vendedor": "Ana L",  "produto": "Teclado",   "categoria": "Periféricos", "qtd": 2, "preco":  150.0, "status": "entregue"},
    {"pedido": 7,  "cliente": "Lucas",   "vendedor": "Carla",  "produto": "Notebook",  "categoria": "Eletrônicos", "qtd": 1, "preco": 4500.0, "status": "pendente"},
    {"pedido": 8,  "cliente": "Pedro",   "vendedor": "Bruno",  "produto": "Mouse",     "categoria": "Periféricos", "qtd": 2, "preco":   89.9, "status": "entregue"},
    {"pedido": 9,  "cliente": "Beatriz", "vendedor": "Diego",  "produto": "SSD 1TB",  "categoria": "Componentes", "qtd": 3, "preco":  380.0, "status": "entregue"},
    {"pedido": 10, "cliente": "Lucas",   "vendedor": "Ana L",  "produto": "Monitor 4K","categoria": "Eletrônicos", "qtd": 2, "preco": 2800.0, "status": "entregue"},
])

vendas

pedido,cliente,vendedor,produto,categoria,qtd,preco,status
i64,str,str,str,str,i64,f64,str
1,"""João""","""Bruno""","""Notebook""","""Eletrônicos""",1,4500.0,"""entregue"""
2,"""Maria""","""Carla""","""Mouse""","""Periféricos""",3,89.9,"""entregue"""
3,"""João""","""Bruno""","""SSD 1TB""","""Componentes""",2,380.0,"""entregue"""
4,"""Ana""","""Diego""","""Headset""","""Periféricos""",1,450.0,"""cancelado"""
5,"""Pedro""","""Bruno""","""Monitor 4K""","""Eletrônicos""",1,2800.0,"""entregue"""
6,"""Maria""","""Ana L""","""Teclado""","""Periféricos""",2,150.0,"""entregue"""
7,"""Lucas""","""Carla""","""Notebook""","""Eletrônicos""",1,4500.0,"""pendente"""
8,"""Pedro""","""Bruno""","""Mouse""","""Periféricos""",2,89.9,"""entregue"""
9,"""Beatriz""","""Diego""","""SSD 1TB""","""Componentes""",3,380.0,"""entregue"""


In [6]:
vendas = pl.DataFrame(vendas)
vendas

pedido,cliente,vendedor,produto,categoria,qtd,preco,status
i64,str,str,str,str,i64,f64,str
1,"""João""","""Bruno""","""Notebook""","""Eletrônicos""",1,4500.0,"""entregue"""
2,"""Maria""","""Carla""","""Mouse""","""Periféricos""",3,89.9,"""entregue"""
3,"""João""","""Bruno""","""SSD 1TB""","""Componentes""",2,380.0,"""entregue"""
4,"""Ana""","""Diego""","""Headset""","""Periféricos""",1,450.0,"""cancelado"""
5,"""Pedro""","""Bruno""","""Monitor 4K""","""Eletrônicos""",1,2800.0,"""entregue"""
6,"""Maria""","""Ana L""","""Teclado""","""Periféricos""",2,150.0,"""entregue"""
7,"""Lucas""","""Carla""","""Notebook""","""Eletrônicos""",1,4500.0,"""pendente"""
8,"""Pedro""","""Bruno""","""Mouse""","""Periféricos""",2,89.9,"""entregue"""
9,"""Beatriz""","""Diego""","""SSD 1TB""","""Componentes""",3,380.0,"""entregue"""


## Q1 — Adicionar coluna `faturamento`

Adicione a coluna `faturamento` (`qtd * preco`) retornando um **novo** DataFrame.  
Em Polars não existe `.assign()` — use o método equivalente com expressões `pl.col()`.

> **Dica:** o método se chama `with_columns()` e recebe expressões Polars.

In [7]:
# Q1 — Criando a coluna de faturamento
vendas = vendas.with_columns(
    faturamento=(pl.col("qtd") * pl.col("preco"))
)
print(vendas)

shape: (10, 9)
┌────────┬─────────┬──────────┬────────────┬───┬─────┬────────┬───────────┬─────────────┐
│ pedido ┆ cliente ┆ vendedor ┆ produto    ┆ … ┆ qtd ┆ preco  ┆ status    ┆ faturamento │
│ ---    ┆ ---     ┆ ---      ┆ ---        ┆   ┆ --- ┆ ---    ┆ ---       ┆ ---         │
│ i64    ┆ str     ┆ str      ┆ str        ┆   ┆ i64 ┆ f64    ┆ str       ┆ f64         │
╞════════╪═════════╪══════════╪════════════╪═══╪═════╪════════╪═══════════╪═════════════╡
│ 1      ┆ João    ┆ Bruno    ┆ Notebook   ┆ … ┆ 1   ┆ 4500.0 ┆ entregue  ┆ 4500.0      │
│ 2      ┆ Maria   ┆ Carla    ┆ Mouse      ┆ … ┆ 3   ┆ 89.9   ┆ entregue  ┆ 269.7       │
│ 3      ┆ João    ┆ Bruno    ┆ SSD 1TB    ┆ … ┆ 2   ┆ 380.0  ┆ entregue  ┆ 760.0       │
│ 4      ┆ Ana     ┆ Diego    ┆ Headset    ┆ … ┆ 1   ┆ 450.0  ┆ cancelado ┆ 450.0       │
│ 5      ┆ Pedro   ┆ Bruno    ┆ Monitor 4K ┆ … ┆ 1   ┆ 2800.0 ┆ entregue  ┆ 2800.0      │
│ 6      ┆ Maria   ┆ Ana L    ┆ Teclado    ┆ … ┆ 2   ┆ 150.0  ┆ entregue  ┆ 300.0    

### 📝 Q1 — Revisão

**Nota: 9/10**

`with_columns()` correto. Dois pontos de atenção:

**1. Sobrescrita do original:**
```python
# ❌ Perde o DataFrame sem faturamento
vendas = vendas.with_columns(...)

# ✅ Preserva o original (padrão imutável — mesmo princípio do .assign() no pandas)
vendas_faturadas = vendas.with_columns(...)
```

**2. Dois estilos de nomear colunas — ambos válidos em Polars:**
```python
# Estilo keyword argument (mais compacto)
vendas.with_columns(faturamento=(pl.col("qtd") * pl.col("preco")))

# Estilo .alias() — canônico Polars, mais explícito
vendas.with_columns((pl.col("qtd") * pl.col("preco")).alias("faturamento"))
```
O estilo `.alias()` é preferido em código de produção por ser mais explícito e funcionar em qualquer posição (inclusive dentro de `.agg()` e `select()`).

## Q2 — Filtrar pedidos relevantes

Filtre apenas os pedidos com `status == "entregue"` **e** `faturamento > 500`.  
Armazene em `vendas_relevantes`.

Em Polars não existe `.query()` — use `.filter()` com `pl.col()`.

> **Dica:** condições múltiplas usam `&` entre expressões `pl.col()`.

In [8]:
# Filtrando status "entregue" e faturamento > 500
vendas_filtrado = vendas.filter(
    (pl.col("status") == "entregue") & 
    (pl.col("faturamento") > 500)
)
print(vendas_filtrado)

shape: (5, 9)
┌────────┬─────────┬──────────┬────────────┬───┬─────┬────────┬──────────┬─────────────┐
│ pedido ┆ cliente ┆ vendedor ┆ produto    ┆ … ┆ qtd ┆ preco  ┆ status   ┆ faturamento │
│ ---    ┆ ---     ┆ ---      ┆ ---        ┆   ┆ --- ┆ ---    ┆ ---      ┆ ---         │
│ i64    ┆ str     ┆ str      ┆ str        ┆   ┆ i64 ┆ f64    ┆ str      ┆ f64         │
╞════════╪═════════╪══════════╪════════════╪═══╪═════╪════════╪══════════╪═════════════╡
│ 1      ┆ João    ┆ Bruno    ┆ Notebook   ┆ … ┆ 1   ┆ 4500.0 ┆ entregue ┆ 4500.0      │
│ 3      ┆ João    ┆ Bruno    ┆ SSD 1TB    ┆ … ┆ 2   ┆ 380.0  ┆ entregue ┆ 760.0       │
│ 5      ┆ Pedro   ┆ Bruno    ┆ Monitor 4K ┆ … ┆ 1   ┆ 2800.0 ┆ entregue ┆ 2800.0      │
│ 9      ┆ Beatriz ┆ Diego    ┆ SSD 1TB    ┆ … ┆ 3   ┆ 380.0  ┆ entregue ┆ 1140.0      │
│ 10     ┆ Lucas   ┆ Ana L    ┆ Monitor 4K ┆ … ┆ 2   ┆ 2800.0 ┆ entregue ┆ 5600.0      │
└────────┴─────────┴──────────┴────────────┴───┴─────┴────────┴──────────┴─────────────┘


### 📝 Q2 — Revisão

**Nota: 10/10**

`.filter()` com `pl.col()` e `&` — correto e idiomático.

**Comparação com pandas e SQL:**
```python
# Pandas
df.query("status == 'entregue' and faturamento > 500")

# Polars
df.filter((pl.col("status") == "entregue") & (pl.col("faturamento") > 500))

# SQL equivalente
WHERE status = 'entregue' AND faturamento > 500
```

**Por que os parênteses são obrigatórios em Polars:**  
Em Python, `&` tem precedência maior que `==`. Sem parênteses:
```python
# ❌ Avalia como: pl.col("status") == ("entregue" & pl.col("faturamento")) > 500
pl.col("status") == "entregue" & pl.col("faturamento") > 500
```
Sempre envolva cada condição em `()`.

## Q3 — Resumo por vendedor

Usando `group_by` em `vendas_relevantes`, calcule por **vendedor**:
- `total_pedidos` — quantidade de pedidos
- `faturamento_total` — soma do faturamento
- `ticket_medio` — média do faturamento

Ordene por `faturamento_total` decrescente.

> **Dica:** em Polars, `.agg()` recebe expressões com `.alias()` para nomear colunas.

In [9]:
# Calculando métricas agregadas
vendas_vendedor = vendas.group_by("vendedor").agg(
    pl.len().alias("total_pedidos"),
    pl.col("faturamento").sum().alias("faturamento_total"),
    pl.col("faturamento").mean().alias("ticket_medio")
).sort("faturamento_total", descending=True)

print(vendas_vendedor)

shape: (4, 4)
┌──────────┬───────────────┬───────────────────┬──────────────┐
│ vendedor ┆ total_pedidos ┆ faturamento_total ┆ ticket_medio │
│ ---      ┆ ---           ┆ ---               ┆ ---          │
│ str      ┆ u32           ┆ f64               ┆ f64          │
╞══════════╪═══════════════╪═══════════════════╪══════════════╡
│ Bruno    ┆ 4             ┆ 8239.8            ┆ 2059.95      │
│ Ana L    ┆ 2             ┆ 5900.0            ┆ 2950.0       │
│ Carla    ┆ 2             ┆ 4769.7            ┆ 2384.85      │
│ Diego    ┆ 2             ┆ 1590.0            ┆ 795.0        │
└──────────┴───────────────┴───────────────────┴──────────────┘


### 📝 Q3 — Revisão

**Nota: 7/10**

Sintaxe perfeita — `pl.len()`, `.alias()`, `.sort()` todos corretos. Porém há um **bug de escopo silencioso**: você usou `vendas` (dados completos, incluindo cancelados e pendentes) em vez de `vendas_filtrado` (apenas entregues com faturamento > 500).

```python
# ❌ Inclui pedido 4 (Headset cancelado, R$450) e pedido 7 (Notebook pendente, R$4500)
vendas.group_by("vendedor").agg(...)

# ✅ Apenas os pedidos relevantes
vendas_filtrado.group_by("vendedor").agg(...)
```

**Por que é perigoso:** o código executa sem erro, os números parecem plausíveis, mas estão errados. Em produção esse tipo de bug passa por code review sem ser detectado facilmente.

**`pl.len()` vs `pl.col("pedido").count()`:**
```python
pl.len()                    # conta todas as linhas do grupo (inclui nulls)
pl.col("pedido").count()    # conta valores não-null na coluna "pedido"
# Para IDs que nunca são null, o resultado é o mesmo — mas pl.len() é mais explícito
```

## Q4 — Participação por categoria

Calcule o `faturamento_total` por **categoria** e adicione a coluna `percentual`.

Em Polars existe uma forma de fazer isso **em uma única expressão**, sem `group_by` separado — usando `over()`, que é a window function nativa do Polars.

> **Dica:** `pl.col("faturamento").sum().over("categoria")` calcula a soma por grupo mantendo todas as linhas (análogo ao `.transform()` do pandas e ao `SUM() OVER (PARTITION BY)` do SQL).

In [11]:
# Calculando o faturamento total por categoria e o percentual representativo
vendas_percentual = vendas.with_columns(
    faturamento_total_categoria=pl.col("faturamento").sum().over("categoria")
).with_columns(
    percentual=(pl.col("faturamento") / pl.col("faturamento_total_categoria")) * 100
)

# Alternativa direta em uma única instrução:
# percentual=(pl.col("faturamento") / pl.col("faturamento").sum().over("categoria")) * 100

print(vendas_percentual)

shape: (10, 11)
┌────────┬─────────┬──────────┬────────────┬───┬───────────┬─────────────┬────────────┬────────────┐
│ pedido ┆ cliente ┆ vendedor ┆ produto    ┆ … ┆ status    ┆ faturamento ┆ faturament ┆ percentual │
│ ---    ┆ ---     ┆ ---      ┆ ---        ┆   ┆ ---       ┆ ---         ┆ o_total_ca ┆ ---        │
│ i64    ┆ str     ┆ str      ┆ str        ┆   ┆ str       ┆ f64         ┆ tegoria    ┆ f64        │
│        ┆         ┆          ┆            ┆   ┆           ┆             ┆ ---        ┆            │
│        ┆         ┆          ┆            ┆   ┆           ┆             ┆ f64        ┆            │
╞════════╪═════════╪══════════╪════════════╪═══╪═══════════╪═════════════╪════════════╪════════════╡
│ 1      ┆ João    ┆ Bruno    ┆ Notebook   ┆ … ┆ entregue  ┆ 4500.0      ┆ 17400.0    ┆ 25.862069  │
│ 2      ┆ Maria   ┆ Carla    ┆ Mouse      ┆ … ┆ entregue  ┆ 269.7       ┆ 1199.5     ┆ 22.484368  │
│ 3      ┆ João    ┆ Bruno    ┆ SSD 1TB    ┆ … ┆ entregue  ┆ 760.0       ┆ 

### 📝 Q4 — Revisão

**Nota: 10/10**

Uso excelente de `over()`. A abordagem em dois `with_columns()` encadeados é mais legível e você ainda documentou a alternativa one-liner — demonstra compreensão do mecanismo.

**`over()` — a window function do Polars:**
```python
pl.col("faturamento").sum().over("categoria")
# Análogos:
# SQL:    SUM(faturamento) OVER (PARTITION BY categoria)
# Pandas: df.groupby("categoria")["faturamento"].transform("sum")
```

**Diferença fundamental `group_by` vs `over()`:**
```
group_by().agg()  → colapsa: 1 linha por grupo
over()            → mantém shape: N linhas, valor do grupo repetido por linha

Exemplo com 3 pedidos na categoria Eletrônicos (R$4500, R$2800, R$5600):
  group_by → 1 linha: Eletrônicos | 12900.0
  over()   → 3 linhas, cada uma com 12900.0 na nova coluna
```

O `over()` é essencial para calcular percentuais, rankings e desvios da média sem perder granularidade das linhas.

## Q5 — Produto de maior faturamento por cliente

Para cada **cliente**, encontre o produto com maior faturamento individual.  
Em Polars não existe `idxmax()` — use `sort()` + `group_by()` com `.first()`, ou qualquer abordagem sem loop `for`.

> **Dica:** se você ordenar o DataFrame por `faturamento` descrescente antes do `group_by`, o `.first()` dentro do `.agg()` vai pegar o maior.

In [14]:
# Ordenando de forma decrescente e pegando o primeiro registro de cada cliente
vendas_top_produto = vendas.sort("faturamento", descending=True).group_by("cliente").agg(
    pl.col("produto").first().alias("produto_maior_faturamento"),
    pl.col("faturamento").first()
)

print(vendas_top_produto)

shape: (6, 3)
┌─────────┬───────────────────────────┬─────────────┐
│ cliente ┆ produto_maior_faturamento ┆ faturamento │
│ ---     ┆ ---                       ┆ ---         │
│ str     ┆ str                       ┆ f64         │
╞═════════╪═══════════════════════════╪═════════════╡
│ Maria   ┆ Teclado                   ┆ 300.0       │
│ Ana     ┆ Headset                   ┆ 450.0       │
│ João    ┆ Notebook                  ┆ 4500.0      │
│ Beatriz ┆ SSD 1TB                   ┆ 1140.0      │
│ Pedro   ┆ Monitor 4K                ┆ 2800.0      │
│ Lucas   ┆ Monitor 4K                ┆ 5600.0      │
└─────────┴───────────────────────────┴─────────────┘


### 📝 Q5 — Revisão

**Nota: 10/10**

Padrão correto — `sort()` descrescente antes do `group_by()` garante que `.first()` pega o maior valor.

**Como funciona internamente:**
```python
vendas
  .sort("faturamento", descending=True)  # ordena globalmente: maior faturamento primeiro
  .group_by("cliente")                   # agrupa — cada grupo já está ordenado
  .agg(pl.col("produto").first())        # first() = primeiro do grupo = maior faturamento
```

**Detalhe: `.alias()` na segunda coluna**
```python
# ✅ Mais explícito em produção
.agg(
    pl.col("produto").first().alias("produto_maior_faturamento"),
    pl.col("faturamento").first().alias("faturamento"),  # mantém nome explícito
)
```

**Comparação com a abordagem Pandas:**
```python
# Pandas — usa índice com idxmax()
idx = df.groupby("cliente")["faturamento"].idxmax()
df.loc[idx, ["cliente", "produto", "faturamento"]]

# Polars — sem índice, usa sort + first
df.sort("faturamento", descending=True).group_by("cliente").agg(
    pl.col("produto").first().alias("produto"),
    pl.col("faturamento").first(),
)
```
Polars não tem índice de linhas — por design. Toda operação é por valor, nunca por posição de memória. Isso torna o Polars mais seguro e previsível em pipelines paralelos.